## Step 1 — Install Dependencies

In [2]:
!pip install pdfplumber spacy pandas -q
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 8.5 MB/s eta 0:00:02
     ------ --------------------------------- 2.1/12.8 MB 7.8 MB/s eta 0:00:02
     ---------- ----------------------------- 3.4/12.8 MB 7.2 MB/s eta 0:00:02
     --------------- ------------------------ 5.0/12.8 MB 7.0 MB/s eta 0:00:02
     -------------------- ------------------- 6.6/12.8 MB 7.1 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 7.2 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 7.2 MB/s eta 0:00:01
     ---------------------------------- ----- 11.0/12.8 MB 7.2 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 7.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 6.8 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the p

## Step 2 — Import Libraries

In [4]:
import pdfplumber
import spacy
import pandas as pd
import os

nlp = spacy.load('en_core_web_sm')
print(' Libraries loaded successfully')

 Libraries loaded successfully


In [5]:
PDF_PATH = 'police_report.pdf'  # <-- change this if your filename is different

if not os.path.exists(PDF_PATH):
    print(f'❌ File not found: {PDF_PATH}')
    print('Please place the PDF in the same folder as this notebook.')
else:
    print(f' Found: {PDF_PATH}')

 Found: police_report.pdf


## Step 4 — Extract Text from PDF

In [7]:
pages_text = []

with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Total pages: {len(pdf.pages)}')
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            pages_text.append({'page': i + 1, 'text': text})

full_text = '\n'.join([p['text'] for p in pages_text])

print(f'\n Extracted text from {len(pages_text)} pages')
print(f'\n--- Preview (first 500 chars) ---\n')
print(full_text[:500])

Total pages: 75

 Extracted text from 14 pages

--- Preview (first 500 chars) ---

To: Whom it may Concern
From: Fort Smith Police Department
Date: January 19, 2015
Ref: MRAP
The following documentation is intended to document the intended use and training of the Mine Resistatant
Ambush Protection Vehicle, which was allocated to the Fort Smith Police Department, by the United States Government.
The Fort Smith Police Department recognizes this vehicle is the Property of the United Stated government and
assigned to this agency, if and when this agency no longer has a need for th


## Step 5 — Run NER (Named Entity Recognition)

In [9]:
# spaCy has a max text length — split if needed
MAX_CHARS = 100000
text_chunk = full_text[:MAX_CHARS]

doc = nlp(text_chunk)

# Extract entities by type
dates     = list(dict.fromkeys([ent.text.strip() for ent in doc.ents if ent.label_ == 'DATE']))
locations = list(dict.fromkeys([ent.text.strip() for ent in doc.ents if ent.label_ in ['GPE', 'LOC', 'FAC']]))
people    = list(dict.fromkeys([ent.text.strip() for ent in doc.ents if ent.label_ == 'PERSON']))
orgs      = list(dict.fromkeys([ent.text.strip() for ent in doc.ents if ent.label_ == 'ORG']))

print('Dates found:',     dates[:5])
print('Locations found:', locations[:5])
print('People found:',    people[:5])
print('Orgs found:',     orgs[:5])

Dates found: ['January 19, 2015', 'June 30th, 2014', 'January 14, 2015', 'January 13, 2015', 'December 2015']
Locations found: ['Rostan', 'the United States\nGovernment', '• Mount', 'Lonoke County Sheriff’s Office', 'BTJhEeC']
People found: ['McMillen', 'LESO Liaison\nCOURSE', 'Training', 'Driver', 'Eric Wacaster']
Orgs found: ['Fort Smith Police Department\nDate', 'the Fort Smith Police Department', 'the United States Government', 'The Fort Smith Police Department', 'the Property of the United Stated']


## Step 6 — Build Structured Output (One Row per Page)

In [11]:
records = []

for i, page_data in enumerate(pages_text):
    page_text = page_data['text']
    page_doc  = nlp(page_text[:5000])  # limit per page to avoid slowness

    page_dates     = [ent.text for ent in page_doc.ents if ent.label_ == 'DATE']
    page_locations = [ent.text for ent in page_doc.ents if ent.label_ in ['GPE', 'LOC', 'FAC']]
    page_people    = [ent.text for ent in page_doc.ents if ent.label_ == 'PERSON']
    page_orgs      = [ent.text for ent in page_doc.ents if ent.label_ == 'ORG']

    # Simple incident type guess from keywords
    text_lower = page_text.lower()
    if any(w in text_lower for w in ['training', 'program', 'equipment']):
        incident_type = 'Training / Equipment'
    elif any(w in text_lower for w in ['theft', 'stolen', 'robbery']):
        incident_type = 'Theft'
    elif any(w in text_lower for w in ['assault', 'attack', 'fight']):
        incident_type = 'Assault'
    elif any(w in text_lower for w in ['accident', 'crash', 'collision']):
        incident_type = 'Traffic Accident'
    elif any(w in text_lower for w in ['fire', 'burn', 'smoke']):
        incident_type = 'Fire'
    else:
        incident_type = 'General Report'

    records.append({
        'Report_ID':     f'RPT_{i+1:03d}',
        'Incident_Type': incident_type,
        'Date':          page_dates[0]     if page_dates     else 'Unknown',
        'Location':      page_locations[0] if page_locations else 'Unknown',
        'Officer':       page_people[0]    if page_people    else 'Unknown',
        'Organization':  page_orgs[0]      if page_orgs      else 'Unknown',
        'Summary':       page_text[:200].replace('\n', ' ').strip()
    })

df_pdf = pd.DataFrame(records)
print(f'Extracted {len(df_pdf)} records')
df_pdf.head()

Extracted 14 records


,Report_ID,Incident_Type,Date,Location,Officer,Organization,Summary
0,RPT_001,Training / Equipment,"January 19, 2015",Unknown,McMillen,Fort Smith Police Department\nDate,To: Whom it may Concern From: Fort Smith Polic...
1,RPT_002,Training / Equipment,"June 30th, 2014",Unknown,Training,Cpl Monty McMillen\n5.0 Hours Swat Team Operat...,COURSE: LESSON TITLE: CLEET Continuing Educati...
2,RPT_003,Training / Equipment,Unknown,Unknown,Unknown,Evacuations / Emergency Situations\nRequired T...,Evacuations / Emergency Situations Required Tr...
3,RPT_004,Training / Equipment,Unknown,Unknown,Eric Wacaster,Hot Springs Police Department,Hot Springs Police Department S.W.A.T. Trainin...
4,RPT_005,Training / Equipment,Unknown,Rostan,Eric Wacaster,Hot Springs Police Department,Hot Springs Police Department S.W.A.T. Trainin...


## Step 7 — Save Output CSV

In [13]:
df_pdf.to_csv('pdf_output.csv', index=False)
print(' Saved: pdf_output.csv')
print(f'   Shape: {df_pdf.shape}')
df_pdf

 Saved: pdf_output.csv
   Shape: (14, 7)


,Report_ID,Incident_Type,Date,Location,Officer,Organization,Summary
0,RPT_001,Training / Equipment,"January 19, 2015",Unknown,McMillen,Fort Smith Police Department\nDate,To: Whom it may Concern From: Fort Smith Polic...
1,RPT_002,Training / Equipment,"June 30th, 2014",Unknown,Training,Cpl Monty McMillen\n5.0 Hours Swat Team Operat...,COURSE: LESSON TITLE: CLEET Continuing Educati...
2,RPT_003,Training / Equipment,Unknown,Unknown,Unknown,Evacuations / Emergency Situations\nRequired T...,Evacuations / Emergency Situations Required Tr...
3,RPT_004,Training / Equipment,Unknown,Unknown,Eric Wacaster,Hot Springs Police Department,Hot Springs Police Department S.W.A.T. Trainin...
4,RPT_005,Training / Equipment,Unknown,Rostan,Eric Wacaster,Hot Springs Police Department,Hot Springs Police Department S.W.A.T. Trainin...
5,RPT_006,Training / Equipment,Unknown,Unknown,Shut Down,Hot Springs Police Department,Hot Springs Police Department S.W.A.T. Trainin...
6,RPT_007,Training / Equipment,Unknown,Unknown,Zac Rostan,Unknown,References None Training Aids MRAP Coordinatio...
7,RPT_008,Training / Equipment,"January 14, 2015",the United States\nGovernment,Brett Hibbs\nDate,the Jacksonville AR Police Department,To: Whom it may Concern From: Lt. Brett Hibbs ...
8,RPT_009,Training / Equipment,Unknown,Unknown,Driver Training Mine Resistant Ambush,Operations of the Mine Resistant Ambush Protec...,COURSE: LESSON TITLE: MRAP Driver Training Min...
9,RPT_010,Training / Equipment,"January 13, 2015",Unknown,Vehicle,Vehicle\nStandard Operating Procedure,Mine Resistant Ambush Protected (MRAP) Vehicle...


## Step 8 — Quick Summary Stats

In [ ]:
import matplotlib.pyplot as plt

print('=== PDF Modality Summary ===')
print(f'Total reports extracted : {len(df_pdf)}')
print(f'Unique incident types   : {df_pdf["Incident_Type"].nunique()}')
print(f'Records with location   : {(df_pdf["Location"] != "Unknown").sum()}')
print(f'Records with date       : {(df_pdf["Date"] != "Unknown").sum()}')
print(f'Records with officer    : {(df_pdf["Officer"] != "Unknown").sum()}')

# Plot incident type distribution
df_pdf['Incident_Type'].value_counts().plot(
    kind='bar', color='steelblue', figsize=(8, 4)
)
plt.title('Incident Types Extracted from PDF')
plt.xlabel('Incident Type')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('pdf_chart.png', dpi=150)
plt.show()
print('Chart saved: pdf_chart.png')